# LeetCode 2670 — Find the Distinct Difference Array bằng Python

Link đề: https://leetcode.com/problems/find-the-distinct-difference-array/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table

Skeleton LeetCode:

```python
class Solution:
    def distinctDifferenceArray(self, nums: List[int]) -> List[int]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu khái niệm **tiền tố (prefix)** `nums[0..i]` và **hậu tố (suffix)** `nums[i+1..n-1]`, kể cả hậu tố **rỗng** ở cuối mảng.
2. Dùng `set` để đếm **số phần tử khác nhau (distinct)**, và `collections.Counter` / `dict` để đếm **số lần xuất hiện** khi cần "bỏ" phần tử ra khỏi tập.
3. Hiểu kỹ thuật **tiền xử lý prefix / suffix**: tính trước một mảng từ trái sang phải và một mảng từ phải sang trái, rồi ghép lại — biến `O(n²)` thành `O(n)`.
4. Viết 5 cách: vét cạn bằng `set` của lát cắt, mảng distinct tiền tố + hậu tố, một `set` tiền tố + một `Counter` hậu tố giảm dần, **bitmask** (vì giá trị `<= 50`), và cách quét theo **vị trí xuất hiện đầu / cuối** của mỗi giá trị.
5. Tránh các lỗi: quên rằng hậu tố của phần tử cuối là rỗng (`0` giá trị), tính hậu tố gồm cả `nums[i]`, xoá khỏi `set` khi còn bản sao ở hậu tố, và sai chỉ số lệch một.

## 2. Đề bài & ví dụ

Cho mảng `nums` độ dài `n` (chỉ số từ 0). **Mảng hiệu khác nhau (distinct difference array)** của `nums` là mảng `diff` độ dài `n` với

`diff[i] = (số phần tử khác nhau của tiền tố nums[0..i]) − (số phần tử khác nhau của hậu tố nums[i+1..n-1])`.

Ký hiệu `nums[i..j]` là đoạn con từ `i` đến `j` (gồm cả hai đầu); nếu `i > j` thì đoạn rỗng. Trả về `diff`.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,2,3,4,5]` | `[-3,-1,1,3,5]` | `i=0`: `1 − 4 = −3`; `i=1`: `2 − 3 = −1`; `i=2`: `3 − 2 = 1`; `i=3`: `4 − 1 = 3`; `i=4`: `5 − 0 = 5`. |
| 2 | `[3,2,3,4,2]` | `[-2,-1,0,2,3]` | `i=0`: tiền tố `{3}` (1), hậu tố `{2,3,4}` (3) ⇒ `−2`; `i=1`: `{3,2}` (2), `{3,4,2}` (3) ⇒ `−1`; `i=2`: `{3,2}` (2), `{4,2}` (2) ⇒ `0`; `i=3`: `{3,2,4}` (3), `{2}` (1) ⇒ `2`; `i=4`: `3 − 0 = 3`. |

Constraints:

- `1 <= n == nums.length <= 50`
- `1 <= nums[i] <= 50`

Chú ý: tiền tố **gồm** `nums[i]`, còn hậu tố bắt đầu từ `i + 1` nên **không gồm** `nums[i]`.

In [1]:
nums = [3, 2, 3, 4, 2]
n = len(nums)
for i in range(n):
    print(f"i={i}: tiền tố {nums[:i + 1]} | hậu tố {nums[i + 1:]}")

i=0: tiền tố [3] | hậu tố [2, 3, 4, 2]
i=1: tiền tố [3, 2] | hậu tố [3, 4, 2]
i=2: tiền tố [3, 2, 3] | hậu tố [4, 2]
i=3: tiền tố [3, 2, 3, 4] | hậu tố [2]
i=4: tiền tố [3, 2, 3, 4, 2] | hậu tố []


## 3. Kiến thức nền

### 3.1. Cắt lát (slicing): tiền tố và hậu tố

- `nums[:i + 1]` là tiền tố gồm các phần tử `nums[0], ..., nums[i]`.
- `nums[i + 1:]` là hậu tố gồm `nums[i + 1], ..., nums[n - 1]`.
- Khi `i = n - 1`: `nums[n:]` là danh sách **rỗng** `[]` (cắt lát không báo lỗi khi vượt biên).

Hai lát cắt này ghép lại thành toàn mảng, không trùng, không thiếu.

In [2]:
nums = [3, 2, 3, 4, 2]
i = 1
print(nums[:i + 1], nums[i + 1:])          # [3, 2] [3, 4, 2]
print(nums[:i + 1] + nums[i + 1:] == nums) # True
print(nums[len(nums):])                    # []: hậu tố rỗng

[3, 2] [3, 4, 2]
True
[]


### 3.2. `set`: đếm phần tử khác nhau

`set(lst)` loại bỏ trùng; `len(set(lst))` là **số giá trị khác nhau**. `set([]) == set()` có độ dài `0` — đúng với hậu tố rỗng.

In [3]:
print(len(set([3, 2, 3])))     # 2
print(len(set([])))            # 0
a = set([3, 2])
a.add(3)                       # thêm phần tử đã có: không đổi
a.add(4)
print(a, len(a))

2
0
{2, 3, 4} 3


### 3.3. `Counter` và `dict`: đếm số lần xuất hiện

`Counter(lst)` cho từ điển `{giá trị: số lần}`. Khi "bỏ" một phần tử khỏi hậu tố, ta giảm đếm đi `1`; nếu về `0` thì giá trị **không còn** trong hậu tố ⇒ xoá khỏi từ điển. Số khoá còn lại chính là số giá trị khác nhau của hậu tố.

In [4]:
from collections import Counter
suffix = Counter([2, 3, 4, 2])
print(suffix, len(suffix))           # Counter({2: 2, 3: 1, 4: 1}) 3
suffix[2] -= 1                       # bỏ một số 2
print(suffix, len(suffix))           # vẫn còn một số 2: 3 giá trị khác nhau
suffix[3] -= 1                       # bỏ số 3: đếm về 0
if suffix[3] == 0:
    del suffix[3]
print(suffix, len(suffix))           # 2 giá trị khác nhau: {2, 4}

Counter({2: 2, 3: 1, 4: 1}) 3
Counter({2: 1, 3: 1, 4: 1}) 3
Counter({2: 1, 4: 1}) 2


### 3.4. Mảng tích luỹ (prefix) và quét từ phải sang trái

Muốn có đáp án cho **mọi** `i` mà không tính lại từ đầu, ta tính một mảng dần dần: `pre[i]` = số giá trị khác nhau trong `nums[0..i]`, tính bằng một `set` thêm từng phần tử. Làm tương tự từ phải sang trái cho hậu tố: `suf[i]` = số giá trị khác nhau trong `nums[i..n-1]`.

In [5]:
nums = [3, 2, 3, 4, 2]
seen, pre = set(), []
for x in nums:
    seen.add(x)
    pre.append(len(seen))
print("pre:", pre)           # [1, 2, 2, 3, 3]

seen, suf = set(), [0] * len(nums)
for i in range(len(nums) - 1, -1, -1):
    seen.add(nums[i])
    suf[i] = len(seen)
print("suf:", suf)           # [3, 3, 3, 2, 1]

pre: [1, 2, 2, 3, 3]
suf: [3, 3, 3, 2, 1]


### 3.5. Bit và `bin(...).count("1")`

Vì `nums[i] <= 50`, ta có thể biểu diễn tập hợp giá trị bằng **một số nguyên** trong đó bit thứ `v` bằng `1` nếu `v` có mặt (bitmask). Thêm `v`: `mask |= 1 << v`. Đếm số giá trị khác nhau = số bit `1` = `bin(mask).count("1")`. (Python 3.10+ có `mask.bit_count()`; `bin(...).count("1")` chạy mọi phiên bản.)

In [6]:
mask = 0
for v in [3, 2, 3, 4, 2]:
    mask |= 1 << v
    print(f"thêm {v}: mask = {bin(mask)}, số giá trị khác nhau = {bin(mask).count('1')}")

thêm 3: mask = 0b1000, số giá trị khác nhau = 1
thêm 2: mask = 0b1100, số giá trị khác nhau = 2
thêm 3: mask = 0b1100, số giá trị khác nhau = 2
thêm 4: mask = 0b11100, số giá trị khác nhau = 3
thêm 2: mask = 0b11100, số giá trị khác nhau = 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Với mỗi vị trí `i`, ta đặt một "vạch" giữa `nums[i]` và `nums[i + 1]`: bên trái vạch là tiền tố, bên phải là hậu tố. Ta đếm xem hai bên có bao nhiêu giá trị **khác nhau** rồi lấy **trái trừ phải**.

Tính tay `nums = [3, 2, 3, 4, 2]`:

```
i=0:  3 | 2 3 4 2      trái {3}=1,     phải {2,3,4}=3   ⇒ -2
i=1:  3 2 | 3 4 2      trái {3,2}=2,   phải {3,4,2}=3   ⇒ -1
i=2:  3 2 3 | 4 2      trái {3,2}=2,   phải {4,2}=2     ⇒  0
i=3:  3 2 3 4 | 2      trái {3,2,4}=3, phải {2}=1       ⇒  2
i=4:  3 2 3 4 2 |      trái {3,2,4}=3, phải {}=0        ⇒  3
```

Quan sát: **khi vạch dịch sang phải một bước, tiền tố nhận thêm `nums[i]`, hậu tố mất đi `nums[i]`**. Nếu ta cập nhật được hai số "distinct" theo từng bước thay vì đếm lại từ đầu, ta tiết kiệm được rất nhiều công.

In [7]:
nums = [3, 2, 3, 4, 2]
n = len(nums)
diff = []
for i in range(n):
    left = len(set(nums[:i + 1]))
    right = len(set(nums[i + 1:]))
    diff.append(left - right)
    print(f"i={i}: trái={left}, phải={right}, hiệu={left - right}")
print(diff)

i=0: trái=1, phải=3, hiệu=-2
i=1: trái=2, phải=3, hiệu=-1
i=2: trái=2, phải=2, hiệu=0
i=3: trái=3, phải=1, hiệu=2
i=4: trái=3, phải=0, hiệu=3
[-2, -1, 0, 2, 3]


## 5. Cách 1 — Brute Force / baseline: `set` của từng lát cắt

**Ý tưởng:** với mỗi `i`, cắt tiền tố `nums[:i + 1]` và hậu tố `nums[i + 1:]`, đổi cả hai thành `set` rồi lấy hiệu độ dài. Làm đúng theo định nghĩa.

In [8]:
from typing import List

class SolutionBrute:
    def distinctDifferenceArray(self, nums: List[int]) -> List[int]:
        n = len(nums)
        diff = []
        for i in range(n):
            left = len(set(nums[:i + 1]))      # tiền tố, gồm nums[i]
            right = len(set(nums[i + 1:]))     # hậu tố, không gồm nums[i]
            diff.append(left - right)
        return diff

In [9]:
sol = SolutionBrute()
print(sol.distinctDifferenceArray([1, 2, 3, 4, 5]))     # [-3, -1, 1, 3, 5]
print(sol.distinctDifferenceArray([3, 2, 3, 4, 2]))     # [-2, -1, 0, 2, 3]
print(sol.distinctDifferenceArray([7]))                 # [1]

[-3, -1, 1, 3, 5]
[-2, -1, 0, 2, 3]
[1]


**Độ phức tạp:** mỗi `i` tạo hai `set` từ tổng `n` phần tử ⇒ Time `O(n²)`; Space `O(n)`. Với `n <= 50` (tối đa vài nghìn phép) vẫn tức thì.

**Nút thắt:** mỗi bước ta **đếm lại từ đầu** cả tiền tố lẫn hậu tố, dù chúng chỉ khác bước trước đúng một phần tử.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá — tách hai nửa độc lập rồi ghép.** Số giá trị khác nhau của **tiền tố** chỉ phụ thuộc phần bên trái; của **hậu tố** chỉ phụ thuộc phần bên phải. Ta có thể:

1. Quét **trái → phải** một lần, giữ `set` các giá trị đã gặp, ghi `pre[i]`.
2. Quét **phải → trái** một lần, giữ `set` tương tự, ghi `suf[i]` (số giá trị khác nhau của `nums[i..n-1]`).
3. Ghép: `diff[i] = pre[i] − suf[i + 1]` (đặt `suf[n] = 0` cho hậu tố rỗng).

Mỗi mảng chỉ tốn một lượt ⇒ tổng `O(n)`.

**Biến thể gọn hơn:** chỉ quét trái → phải, giữ `set` tiền tố **và** `Counter` hậu tố. Mỗi bước: thêm `nums[i]` vào `set`; giảm `Counter[nums[i]]` và nếu về `0` thì xoá; `diff = len(set) − len(Counter)`.

In [10]:
nums = [3, 2, 3, 4, 2]
n = len(nums)
pre = [0] * n
seen = set()
for i, x in enumerate(nums):
    seen.add(x)
    pre[i] = len(seen)
suf = [0] * (n + 1)                  # suf[n] = 0: hậu tố rỗng
seen = set()
for i in range(n - 1, -1, -1):
    seen.add(nums[i])
    suf[i] = len(seen)
print("pre =", pre)
print("suf =", suf)
print("diff =", [pre[i] - suf[i + 1] for i in range(n)])

pre = [1, 2, 2, 3, 3]
suf = [3, 3, 3, 2, 1, 0]
diff = [-2, -1, 0, 2, 3]


## 7. Các cách tiếp theo

### Cách 2 — Hai mảng distinct: tiền tố và hậu tố

Chính là mục 6. Các đoạn: (1) `pre` từ trái sang phải; (2) `suf` có `n + 1` phần tử với `suf[n] = 0`; (3) `diff[i] = pre[i] − suf[i + 1]`.

In [11]:
class SolutionPrefixSuffix:
    def distinctDifferenceArray(self, nums: List[int]) -> List[int]:
        n = len(nums)
        pre = [0] * n
        seen = set()
        for i in range(n):
            seen.add(nums[i])
            pre[i] = len(seen)
        suf = [0] * (n + 1)          # suf[n] = 0: hậu tố rỗng
        seen = set()
        for i in range(n - 1, -1, -1):
            seen.add(nums[i])
            suf[i] = len(seen)
        return [pre[i] - suf[i + 1] for i in range(n)]

print(SolutionPrefixSuffix().distinctDifferenceArray([1, 2, 3, 4, 5]))     # [-3, -1, 1, 3, 5]
print(SolutionPrefixSuffix().distinctDifferenceArray([3, 2, 3, 4, 2]))     # [-2, -1, 0, 2, 3]

[-3, -1, 1, 3, 5]
[-2, -1, 0, 2, 3]


**Độ phức tạp:** hai lượt duyệt, mỗi `add`/`len` trung bình `O(1)` ⇒ Time `O(n)`; Space `O(n)` (hai mảng và các `set`).

### Cách 3 — Một `set` tiền tố + một `Counter` hậu tố (cách nộp)

Khởi tạo `suffix = Counter(nums)` (hậu tố là **toàn mảng**). Duyệt `i` từ trái sang phải:

1. `prefix.add(nums[i])` — tiền tố nhận thêm `nums[i]`.
2. `suffix[nums[i]] -= 1`; nếu về `0` thì `del` — hậu tố mất `nums[i]`, giờ chính là `nums[i + 1..]`.
3. `diff.append(len(prefix) − len(suffix))`.

Thứ tự bước 1–2 trước bước 3 đảm bảo cả hai bên đã cập nhật **cùng vạch** `i`.

In [12]:
from collections import Counter

class SolutionCounter:
    def distinctDifferenceArray(self, nums: List[int]) -> List[int]:
        prefix = set()
        suffix = Counter(nums)
        diff = []
        for x in nums:
            prefix.add(x)
            suffix[x] -= 1
            if suffix[x] == 0:
                del suffix[x]
            diff.append(len(prefix) - len(suffix))
        return diff

print(SolutionCounter().distinctDifferenceArray([1, 2, 3, 4, 5]))     # [-3, -1, 1, 3, 5]
print(SolutionCounter().distinctDifferenceArray([3, 2, 3, 4, 2]))     # [-2, -1, 0, 2, 3]

[-3, -1, 1, 3, 5]
[-2, -1, 0, 2, 3]


**Độ phức tạp:** Time `O(n)` (một lượt, thao tác trung bình `O(1)`); Space `O(n)` cho `set` và `Counter` (tối đa số giá trị khác nhau `<= 50` ở bài này).

### Cách 4 — Bitmask (vì `nums[i] <= 50`)

Biểu diễn tập tiền tố bằng một số nguyên `mask` và các tập hậu tố bằng mảng mask tính từ phải sang trái. Đếm bằng `bin(mask).count("1")`. Cách này **chỉ hợp lệ** khi giá trị nhỏ (ở đây `<= 50`; Python có số nguyên lớn tuỳ ý nên vẫn đúng, nhưng với giá trị lớn mask rất to và không có lợi).

In [13]:
class SolutionBitmask:
    def distinctDifferenceArray(self, nums: List[int]) -> List[int]:
        n = len(nums)
        suffix_mask = [0] * (n + 1)        # suffix_mask[i] = tập giá trị của nums[i..n-1]
        for i in range(n - 1, -1, -1):
            suffix_mask[i] = suffix_mask[i + 1] | (1 << nums[i])
        diff = []
        prefix_mask = 0
        for i in range(n):
            prefix_mask |= 1 << nums[i]
            diff.append(bin(prefix_mask).count("1") - bin(suffix_mask[i + 1]).count("1"))
        return diff

print(SolutionBitmask().distinctDifferenceArray([1, 2, 3, 4, 5]))     # [-3, -1, 1, 3, 5]
print(SolutionBitmask().distinctDifferenceArray([3, 2, 3, 4, 2]))     # [-2, -1, 0, 2, 3]

[-3, -1, 1, 3, 5]
[-2, -1, 0, 2, 3]


**Độ phức tạp:** Time `O(n · V)` nếu đếm bit tốn `O(V)` (với `V = 51` bit, coi như hằng) ⇒ `O(n)`; Space `O(n)` cho mảng mask.

### Cách 5 — Quét theo vị trí xuất hiện đầu tiên và cuối cùng

Một giá trị `x` góp **1** vào số distinct của tiền tố `nums[0..i]` khi và chỉ khi `first[x] <= i` (đã xuất hiện); và góp **1** vào hậu tố `nums[i+1..]` khi và chỉ khi `last[x] > i` (còn xuất hiện ở bên phải). Vậy

`diff[i] = #{x : first[x] ≤ i} − #{x : last[x] > i}`.

Ta đếm bằng hai mảng tần suất theo vị trí: `starts[p]` = số giá trị có `first = p`, `ends[p]` = số giá trị có `last = p`; rồi dùng cộng dồn. Số giá trị có `last[x] > i` bằng `distinct_total − (số giá trị có last ≤ i)`.

In [14]:
class SolutionFirstLast:
    def distinctDifferenceArray(self, nums: List[int]) -> List[int]:
        n = len(nums)
        first, last = {}, {}
        for i, x in enumerate(nums):
            if x not in first:
                first[x] = i          # lần đầu xuất hiện
            last[x] = i               # luôn ghi đè -> lần cuối xuất hiện
        total = len(first)
        starts = [0] * n
        ends = [0] * n
        for x in first:
            starts[first[x]] += 1
            ends[last[x]] += 1
        diff = []
        pre_cnt = 0                   # số giá trị có first <= i
        last_cnt = 0                  # số giá trị có last <= i
        for i in range(n):
            pre_cnt += starts[i]
            last_cnt += ends[i]
            suffix_distinct = total - last_cnt     # số giá trị có last > i
            diff.append(pre_cnt - suffix_distinct)
        return diff

print(SolutionFirstLast().distinctDifferenceArray([1, 2, 3, 4, 5]))     # [-3, -1, 1, 3, 5]
print(SolutionFirstLast().distinctDifferenceArray([3, 2, 3, 4, 2]))     # [-2, -1, 0, 2, 3]

[-3, -1, 1, 3, 5]
[-2, -1, 0, 2, 3]


**Độ phức tạp:** Time `O(n)` (ba lượt đơn giản); Space `O(n)`. Cách này nhìn bài toán theo "đời sống" của mỗi giá trị — từ lần xuất hiện đầu đến lần cuối — và là ý tưởng nền của nhiều bài "số phần tử khác nhau trên mọi khoảng".

## 8. TRACE từng bước

Trace Cách 3 (set + Counter) trên `nums = [3, 2, 3, 4, 2]`. Ban đầu `suffix = {3:2, 2:2, 4:1}` (đếm toàn mảng), `prefix = {}`.

| `i` | `x` | `prefix` sau thêm | `suffix` sau bỏ `x` | `len(prefix)` | `len(suffix)` | `diff[i]` |
|---|---|---|---|---|---|---|
| 0 | 3 | `{3}` | `{3:1, 2:2, 4:1}` | 1 | 3 | −2 |
| 1 | 2 | `{3,2}` | `{3:1, 2:1, 4:1}` | 2 | 3 | −1 |
| 2 | 3 | `{3,2}` | `{2:1, 4:1}` (đếm của 3 về 0 ⇒ xoá) | 2 | 2 | 0 |
| 3 | 4 | `{3,2,4}` | `{2:1}` (4 về 0 ⇒ xoá) | 3 | 1 | 2 |
| 4 | 2 | `{3,2,4}` | `{}` (2 về 0 ⇒ xoá) | 3 | 0 | 3 |

Kết quả `[-2, -1, 0, 2, 3]`.

In [15]:
def trace_counter(nums):
    prefix = set()
    suffix = Counter(nums)
    print("ban đầu: prefix =", set(prefix), "| suffix =", dict(suffix))
    diff = []
    for i, x in enumerate(nums):
        prefix.add(x)
        suffix[x] -= 1
        note = ""
        if suffix[x] == 0:
            del suffix[x]
            note = f"  (đếm của {x} về 0 -> xoá)"
        d = len(prefix) - len(suffix)
        diff.append(d)
        print(f"i={i} x={x}: prefix={sorted(prefix)} suffix={dict(suffix)}{note} -> {len(prefix)} - {len(suffix)} = {d}")
    print("kết quả:", diff)
    return diff

trace_counter([3, 2, 3, 4, 2])

ban đầu: prefix = set() | suffix = {3: 2, 2: 2, 4: 1}
i=0 x=3: prefix=[3] suffix={3: 1, 2: 2, 4: 1} -> 1 - 3 = -2
i=1 x=2: prefix=[2, 3] suffix={3: 1, 2: 1, 4: 1} -> 2 - 3 = -1
i=2 x=3: prefix=[2, 3] suffix={2: 1, 4: 1}  (đếm của 3 về 0 -> xoá) -> 2 - 2 = 0
i=3 x=4: prefix=[2, 3, 4] suffix={2: 1}  (đếm của 4 về 0 -> xoá) -> 3 - 1 = 2
i=4 x=2: prefix=[2, 3, 4] suffix={}  (đếm của 2 về 0 -> xoá) -> 3 - 0 = 3
kết quả: [-2, -1, 0, 2, 3]


[-2, -1, 0, 2, 3]

In [16]:
def trace_prefix_suffix(nums):
    n = len(nums)
    pre, seen = [], set()
    for x in nums:
        seen.add(x)
        pre.append(len(seen))
    suf, seen = [0] * (n + 1), set()
    for i in range(n - 1, -1, -1):
        seen.add(nums[i])
        suf[i] = len(seen)
    print("pre =", pre)
    print("suf =", suf, "(suf[n] = 0: hậu tố rỗng)")
    for i in range(n):
        print(f"i={i}: pre[{i}]={pre[i]} - suf[{i + 1}]={suf[i + 1]} = {pre[i] - suf[i + 1]}")

trace_prefix_suffix([3, 2, 3, 4, 2])

pre = [1, 2, 2, 3, 3]
suf = [3, 3, 3, 2, 1, 0] (suf[n] = 0: hậu tố rỗng)
i=0: pre[0]=1 - suf[1]=3 = -2
i=1: pre[1]=2 - suf[2]=3 = -1
i=2: pre[2]=2 - suf[3]=2 = 0
i=3: pre[3]=3 - suf[4]=1 = 2
i=4: pre[4]=3 - suf[5]=0 = 3


## 9. Code nộp LeetCode

Dùng Cách 3 (`set` + `Counter`): một lượt `O(n)`, dễ nhớ.

In [17]:
from collections import Counter
from typing import List

class Solution:
    def distinctDifferenceArray(self, nums: List[int]) -> List[int]:
        prefix = set()
        suffix = Counter(nums)
        diff = []
        for x in nums:
            prefix.add(x)
            suffix[x] -= 1
            if suffix[x] == 0:
                del suffix[x]
            diff.append(len(prefix) - len(suffix))
        return diff

In [18]:
print(Solution().distinctDifferenceArray([1, 2, 3, 4, 5]))     # [-3, -1, 1, 3, 5]
print(Solution().distinctDifferenceArray([3, 2, 3, 4, 2]))     # [-2, -1, 0, 2, 3]

[-3, -1, 1, 3, 5]
[-2, -1, 0, 2, 3]


## 10. Vì sao thuật toán đúng

**Invariant của Cách 3.** Ngay sau khi xử lý phần tử `nums[i]` (tức hết vòng lặp thứ `i`):

- `prefix` đúng bằng tập các giá trị của `nums[0..i]`.
- `suffix` là từ điển đếm số lần xuất hiện của từng giá trị trong `nums[i + 1..n − 1]`, **không chứa khoá có đếm bằng `0`**.

*Khởi tạo:* trước vòng lặp `prefix` rỗng (tiền tố rỗng), `suffix` = đếm toàn mảng = `nums[0..n − 1]`, tức đúng với "`i = −1`".

*Bước:* xử lý `x = nums[i]`: thêm `x` vào `prefix` ⇒ đúng tập `nums[0..i]`. Giảm đếm của `x` đi `1` ⇒ đúng đếm của `nums[i + 1..]` (đã bỏ đúng một bản `x`, chính là `nums[i]`); nếu đếm về `0` thì không còn `x` ở hậu tố nên xoá khoá ⇒ giữ tính chất "không có khoá đếm `0`".

*Kết luận:* `len(prefix)` là số giá trị khác nhau của tiền tố; `len(suffix)` là số khoá còn lại tức số giá trị khác nhau của hậu tố; hiệu của chúng đúng `diff[i]`. Ở `i = n − 1`, `suffix` rỗng nên `diff = len(prefix) − 0`, đúng với hậu tố rỗng.

**Cách 2** đúng vì `pre[i]` và `suf[i + 1]` là đúng hai đại lượng cần; **Cách 5** đúng vì giá trị `x` có mặt ở tiền tố `0..i` ⟺ `first[x] ≤ i` và có mặt ở hậu tố `i+1..` ⟺ `last[x] ≥ i + 1`.

## 11. Phân tích Time / Space Complexity

`n = len(nums)`, `u` = số giá trị khác nhau (`u <= min(n, 50)`).

| Cách | Time | Space |
|---|---|---|
| 1. `set` của từng lát cắt | `O(n²)` | `O(n)` |
| 2. Hai mảng distinct | `O(n)` | `O(n)` |
| 3. `set` + `Counter` | `O(n)` | `O(u)` cho cấu trúc đếm (+ `O(n)` kết quả) |
| 4. Bitmask | `O(n · V)` (`V` = số bit, coi hằng) | `O(n)` |
| 5. Vị trí đầu / cuối | `O(n)` | `O(n)` |

(`add`, `in`, và truy cập `dict` được coi là `O(1)` trung bình.) Với `n <= 50`, mọi cách đều tức thì; sự khác biệt thể hiện khi `n` lớn (`10^5` trở lên): Cách 1 không chạy nổi.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: một phần tử; hai phần tử bằng nhau / khác nhau; toàn phần tử bằng nhau; mọi phần tử khác nhau; giá trị trùng đầu–cuối; giá trị biên `1` và `50`.

In [19]:
sols = [SolutionBrute(), SolutionPrefixSuffix(), SolutionCounter(), SolutionBitmask(), SolutionFirstLast(), Solution()]
cases = [
    ("ví dụ 1", [1, 2, 3, 4, 5], [-3, -1, 1, 3, 5]),
    ("ví dụ 2", [3, 2, 3, 4, 2], [-2, -1, 0, 2, 3]),
    ("một phần tử", [7], [1]),
    ("hai phần tử khác nhau", [1, 2], [0, 2]),
    ("hai phần tử bằng nhau", [5, 5], [0, 1]),
    ("toàn bằng nhau", [4, 4, 4, 4], [0, 0, 0, 1]),
    ("trùng đầu–cuối", [1, 2, 1], [-1, 1, 2]),
    ("giá trị biên", [1, 50, 1, 50], [-1, 0, 1, 2]),
    ("[1,1,2]", [1, 1, 2], [-1, 0, 2]),
]
for name, nums, exp in cases:
    for s in sols:
        got = s.distinctDifferenceArray(nums[:])
        assert got == exp, (name, type(s).__name__, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> [-3, -1, 1, 3, 5]
PASS ví dụ 2 -> [-2, -1, 0, 2, 3]
PASS một phần tử -> [1]
PASS hai phần tử khác nhau -> [0, 2]
PASS hai phần tử bằng nhau -> [0, 1]
PASS toàn bằng nhau -> [0, 0, 0, 1]
PASS trùng đầu–cuối -> [-1, 1, 2]
PASS giá trị biên -> [-1, 0, 1, 2]
PASS [1,1,2] -> [-1, 0, 2]


Kiểm tra tay vài ca:

- `[1, 2]`: `i=0`: tiền tố `{1}` (1), hậu tố `{2}` (1) ⇒ `0`; `i=1`: `2 − 0 = 2` ⇒ `[0, 2]`.
- `[5, 5]`: `i=0`: `1 − 1 = 0`; `i=1`: `1 − 0 = 1` ⇒ `[0, 1]`.
- `[1, 2, 1]`: `i=0`: `1 − |{2,1}| = −1`; `i=1`: `2 − |{1}| = 1`; `i=2`: `2 − 0 = 2`.
- `[1, 50, 1, 50]`: `i=0`: `1 − 2 = −1`; `i=1`: `2 − 2 = 0` (hậu tố `[1,50]`); `i=2`: `2 − 1 = 1`; `i=3`: `2 − 0 = 2`.

### 12.2. Phản ví dụ cho lời giải sai: xoá khỏi `set` hậu tố ngay khi gặp

Lỗi hay gặp (mục 13) là dùng `set` thay cho `Counter` ở hậu tố và `discard` khi duyệt. Ca có giá trị lặp bắt được lỗi này:

In [20]:
def sai_set_suffix(nums):
    prefix, suffix = set(), set(nums)
    diff = []
    for x in nums:
        prefix.add(x)
        suffix.discard(x)         # SAI: x có thể còn xuất hiện ở bên phải
        diff.append(len(prefix) - len(suffix))
    return diff

nums = [3, 2, 3, 4, 2]
print("sai :", sai_set_suffix(nums))
print("đúng:", Solution().distinctDifferenceArray(nums))
assert sai_set_suffix(nums) != Solution().distinctDifferenceArray(nums)
assert sai_set_suffix([1, 2, 3, 4, 5]) == Solution().distinctDifferenceArray([1, 2, 3, 4, 5])   # mảng không trùng thì không lộ lỗi

sai : [-1, 1, 1, 3, 3]
đúng: [-2, -1, 0, 2, 3]


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh `nums` ngẫu nhiên (`1 <= n <= 50`, giá trị `1..50`) với nhiều độ trùng khác nhau (khoảng giá trị nhỏ ⇒ nhiều trùng). So các cách với oracle viết thẳng theo định nghĩa (tập hợp của lát cắt tường minh).

In [21]:
import random
random.seed(2670)

def oracle(nums):
    n = len(nums)
    out = []
    for i in range(n):
        left = set()
        for j in range(0, i + 1):
            left.add(nums[j])
        right = set()
        for j in range(i + 1, n):
            right.add(nums[j])
        out.append(len(left) - len(right))
    return out

for _ in range(500):
    n = random.randint(1, 50)
    hi = random.choice([1, 2, 5, 20, 50])
    nums = [random.randint(1, hi) for _ in range(n)]
    exp = oracle(nums)
    for s in sols:
        got = s.distinctDifferenceArray(nums[:])
        assert got == exp, (type(s).__name__, nums, got, exp)
    assert len(exp) == n and exp[-1] == len(set(nums))
print("500 mảng ngẫu nhiên: mọi cách khớp oracle")

500 mảng ngẫu nhiên: mọi cách khớp oracle


### 12.4. Tính chất cần nhớ

Hai tính chất giúp phát hiện lỗi nhanh: (a) `diff[n − 1] = số giá trị khác nhau của cả mảng`; (b) `diff[0] = 1 − (số giá trị khác nhau của nums[1..])`. Và nếu mọi phần tử đôi một khác nhau thì `diff[i] = (i + 1) − (n − 1 − i) = 2i + 2 − n`.

In [22]:
for n in range(1, 20):
    nums = list(range(1, n + 1))
    assert Solution().distinctDifferenceArray(nums) == [2 * i + 2 - n for i in range(n)]
print("mảng không trùng: diff[i] = 2i + 2 - n đúng với n = 1..19")

mảng không trùng: diff[i] = 2i + 2 - n đúng với n = 1..19


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `set` cho hậu tố rồi `discard` ngay khi gặp phần tử

In [23]:
from typing import List

def sai_discard(nums: List[int]) -> List[int]:
    prefix, suffix = set(), set(nums)
    out = []
    for x in nums:
        prefix.add(x)
        suffix.discard(x)
        out.append(len(prefix) - len(suffix))
    return out

nums = [3, 2, 3, 4, 2]
print("sai :", sai_discard(nums))                       # kết quả lệch so với đáp án đúng
print("đúng:", Solution().distinctDifferenceArray(nums))

sai : [-1, 1, 1, 3, 3]
đúng: [-2, -1, 0, 2, 3]


**Vì sao sai:** `set` không nhớ **số lần** xuất hiện. Ở `i = 0`, `x = 3` còn xuất hiện ở `i = 2`, nhưng `discard(3)` đã coi như hậu tố hết `3`. Cần `Counter` (đếm) và chỉ xoá khi đếm về `0`, hoặc tính trước mảng hậu tố như Cách 2.

### Lỗi 2: Hậu tố tính cả `nums[i]` (lệch một chỉ số)

In [24]:
def sai_hau_to_gom_i(nums: List[int]) -> List[int]:
    n = len(nums)
    return [len(set(nums[:i + 1])) - len(set(nums[i:])) for i in range(n)]    # nums[i:] thay vì nums[i + 1:]

nums = [3, 2, 3, 4, 2]
print("sai :", sai_hau_to_gom_i(nums))
print("đúng:", Solution().distinctDifferenceArray(nums))

sai : [-2, -1, -1, 1, 2]
đúng: [-2, -1, 0, 2, 3]


**Vì sao sai:** đề định nghĩa hậu tố là `nums[i + 1..n − 1]`, **không gồm** `nums[i]`. Dùng `nums[i:]` khiến hậu tố to hơn một phần tử, và ở `i = n − 1` hậu tố có `1` thay vì `0` phần tử (mọi kết quả cuối bị trừ thêm `1`).

### Lỗi 3: Quên hậu tố rỗng khi `i = n − 1`

In [25]:
def sai_quen_rong(nums: List[int]) -> List[int]:
    n = len(nums)
    pre, seen = [], set()
    for x in nums:
        seen.add(x)
        pre.append(len(seen))
    suf, seen = [0] * n, set()
    for i in range(n - 1, -1, -1):
        seen.add(nums[i])
        suf[i] = len(seen)
    out = []
    for i in range(n):
        out.append(pre[i] - (suf[i + 1] if i + 1 < n else suf[i]))   # SAI: lấy suf[i] khi hết mảng
    return out

nums = [3, 2, 3, 4, 2]
print("sai :", sai_quen_rong(nums))
print("đúng:", Solution().distinctDifferenceArray(nums))

sai : [-2, -1, 0, 2, 2]
đúng: [-2, -1, 0, 2, 3]


**Vì sao sai:** với `i = n − 1` hậu tố là **rỗng** nên số giá trị khác nhau là `0`, không phải `suf[n − 1]`. Cách sạch nhất: cấp `suf` có `n + 1` phần tử với `suf[n] = 0` như Cách 2.

### Lỗi 4: Đảo dấu (phải trừ trái)

In [26]:
def sai_dao_dau(nums: List[int]) -> List[int]:
    return [-x for x in Solution().distinctDifferenceArray(nums)]

print("sai :", sai_dao_dau([1, 2, 3, 4, 5]))          # [3, 1, -1, -3, -5]
print("đúng:", Solution().distinctDifferenceArray([1, 2, 3, 4, 5]))   # [-3, -1, 1, 3, 5]

sai : [3, 1, -1, -3, -5]
đúng: [-3, -1, 1, 3, 5]


**Vì sao sai:** đề: **tiền tố trừ hậu tố** (`prefix − suffix`). Ở ví dụ 1 phần tử đầu là `−3` (tiền tố `1`, hậu tố `4`). Hãy tự tính một phần tử bằng tay để chắc chắn về dấu.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề liên quan đến "tiền tố ... hậu tố ... tại mỗi vị trí" ⇒ **tính trước mảng prefix và suffix**, hoặc dịch vạch một bước và cập nhật hai phía.
- Cần "số phần tử khác nhau" ⇒ `set`. Cần "bỏ một phần tử ra khỏi tập có thể còn bản sao" ⇒ `Counter` (đếm) và xoá khi về `0`.
- Giá trị nhỏ (`<= 50`, `<= 64`) ⇒ nghĩ tới **bitmask**.
- Hậu tố/tiền tố **rỗng** luôn là trường hợp biên cần kiểm tra: `n + 1` phần tử với đầu/cuối bằng `0` giúp tránh `if`.
- Với `n` nhỏ, viết **vét cạn** (Cách 1) trước để làm oracle, rồi tối ưu và đối chiếu.

## 15. Biến thể của bài

1. **Number of Good Ways to Split a String (LeetCode 1525):** đếm các vị trí cắt mà số ký tự khác nhau hai bên **bằng nhau** — cùng kỹ thuật prefix/suffix distinct.
2. **Maximum Score After Splitting a String (LeetCode 1422):** tiền tố/hậu tố với số `0` và số `1`.
3. **Product of Array Except Self (LeetCode 238):** prefix và suffix nhân, không dùng chia.
4. **Subarrays with K Different Integers (LeetCode 992):** distinct trên mọi đoạn con, cần cửa sổ trượt.
5. **Distinct difference trên chuỗi ký tự:** `nums` là chuỗi; `set` và `Counter` hoạt động y như cũ.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `set` của lát cắt | Đếm lại từ đầu mỗi `i` | `O(n²)` | `O(n)` | Oracle / `n` rất nhỏ |
| 2. Hai mảng distinct | Prefix và suffix tính trước | `O(n)` | `O(n)` | Rõ ràng, dễ kiểm chứng |
| 3. `set` + `Counter` | Một lượt, cập nhật hai phía | `O(n)` | `O(u)` | **Nên nộp** |
| 4. Bitmask | Tập giá trị là một số nguyên | `O(n · V)` | `O(n)` | Giá trị nhỏ (`<= 50`, `<= 64`) |
| 5. Vị trí đầu / cuối | Mỗi giá trị góp theo đoạn `[first, last)` | `O(n)` | `O(n)` | Khi muốn nhìn theo "đời sống" giá trị |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `len(set([3, 2, 3]))`, `len(set([]))`, `[3, 2, 3, 4, 2][5:]`, `bin(0b1011).count("1")`.

**Bài 2 (tính tay).** Tính `distinctDifferenceArray([1, 1, 2])`: với mỗi `i` viết tiền tố, hậu tố, số khác nhau của mỗi bên và hiệu.

**Bài 3 (điền TODO).** Hoàn thành `suffix_distinct_todo(nums)` trả mảng `suf` độ dài `n + 1` với `suf[i]` là số giá trị khác nhau của `nums[i:]` (`suf[n] = 0`).

In [27]:
def suffix_distinct_todo(nums):
    n = len(nums)
    suf = [0] * (n + 1)
    seen = set()
    # TODO 1: duyệt i từ n - 1 xuống 0
    # TODO 2: thêm nums[i] vào seen
    # TODO 3: suf[i] = len(seen)
    return suf

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `distinct_difference_two_arrays(nums)` theo Cách 2 (hai mảng `pre`, `suf`) mà không nhìn code mẫu. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_balanced_splits(nums)` đếm số vị trí cắt `i` (`0 <= i < n - 1`) mà số giá trị khác nhau của `nums[:i + 1]` **bằng** số giá trị khác nhau của `nums[i + 1:]`. (b) Viết lại `Solution` cho **chuỗi** `s` (trả danh sách các hiệu), kiểm tra `"abca"`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `{3, 2}` có 2 phần tử; `set([])` rỗng; cắt lát vượt biên cho `[]`; `0b1011` có ba bit `1`. Bài 2: tại `i = 0`: tiền tố `[1]`, hậu tố `[1, 2]`.
- **Mức 2 (vừa):** Bài 3: `for i in range(n - 1, -1, -1): seen.add(nums[i]); suf[i] = len(seen)`. Bài 4: `diff[i] = pre[i] - suf[i + 1]`.
- **Mức 3 (sâu):** Bài 5(a): tính `diff` rồi đếm số `i` trong `0..n−2` mà tiền tố bằng hậu tố — tức cần `pre[i] == suf[i + 1]` (đừng dùng `diff == 0` vì `diff[n − 1]` chỉ là tiền tố trừ rỗng). Bài 5(b): `Counter(s)` và `set()` hoạt động trên ký tự; chuỗi cũng duyệt được bằng `for ch in s`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert len(set([3, 2, 3])) == 2
assert len(set([])) == 0
assert [3, 2, 3, 4, 2][5:] == []
assert bin(0b1011).count("1") == 3

# Bài 2
nums = [1, 1, 2]
assert [len(set(nums[:i + 1])) for i in range(3)] == [1, 1, 2]
assert [len(set(nums[i + 1:])) for i in range(3)] == [2, 1, 0]
assert Solution().distinctDifferenceArray(nums) == [-1, 0, 2]

# Bài 3
def suffix_distinct_ans(nums):
    n = len(nums)
    suf = [0] * (n + 1)
    seen = set()
    for i in range(n - 1, -1, -1):
        seen.add(nums[i])
        suf[i] = len(seen)
    return suf

assert suffix_distinct_ans([3, 2, 3, 4, 2]) == [3, 3, 3, 2, 1, 0]

# Bài 4
def distinct_difference_two_arrays(nums):
    n = len(nums)
    pre, seen = [], set()
    for x in nums:
        seen.add(x)
        pre.append(len(seen))
    suf = suffix_distinct_ans(nums)
    return [pre[i] - suf[i + 1] for i in range(n)]

assert distinct_difference_two_arrays([1, 2, 3, 4, 5]) == [-3, -1, 1, 3, 5]
assert distinct_difference_two_arrays([3, 2, 3, 4, 2]) == [-2, -1, 0, 2, 3]

# Bài 5
def count_balanced_splits(nums):
    n = len(nums)
    pre, seen = [], set()
    for x in nums:
        seen.add(x)
        pre.append(len(seen))
    suf = suffix_distinct_ans(nums)
    return sum(1 for i in range(n - 1) if pre[i] == suf[i + 1])

assert count_balanced_splits([1, 2, 2, 1]) == 1       # cắt sau chỉ số 1: {1,2} và {2,1}
assert count_balanced_splits([1, 1, 1]) == 2          # cả hai chỗ cắt đều 1 = 1
assert count_balanced_splits([1, 2, 3]) == 0
assert count_balanced_splits([7]) == 0

def distinct_difference_string(s):
    return Solution().distinctDifferenceArray(list(s))

assert distinct_difference_string("abca") == [1 - 3, 2 - 2, 3 - 1, 3 - 0]   # [-2, 0, 2, 3]
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (`set` cho tiền tố + `Counter` cho hậu tố, cập nhật rồi ghi kết quả)

```python
from collections import Counter
from typing import List

class Solution:
    def distinctDifferenceArray(self, nums: List[int]) -> List[int]:
        prefix = set()
        suffix = Counter(nums)            # ban đầu hậu tố = cả mảng
        diff = []
        for x in nums:
            prefix.add(x)                 # tiền tố nhận x
            suffix[x] -= 1                # hậu tố mất x
            if suffix[x] == 0:
                del suffix[x]
            diff.append(len(prefix) - len(suffix))
        return diff
```

**Checklist**

- [ ] Tiền tố **gồm** `nums[i]`, hậu tố **không gồm** `nums[i]` (bắt đầu từ `i + 1`)?
- [ ] Hậu tố của phần tử cuối là **rỗng** (`0` giá trị)?
- [ ] Dùng `Counter` (không dùng `set` rồi `discard`) cho hậu tố, và xoá khoá khi đếm về `0`?
- [ ] Hiệu là **tiền tố − hậu tố** (đúng dấu)?
- [ ] Đã thử `n = 1`, toàn bằng nhau, toàn khác nhau, giá trị trùng đầu–cuối?
- [ ] Signature đúng `distinctDifferenceArray(self, nums: List[int]) -> List[int]` và có `from typing import List`?